In [2]:
import pandas as pd


In [5]:
df = pd.read_csv( r'C:\Users\DELL SSD\OneDrive\Desktop\customer-trends-data-analysis-End-to-end\customer_shopping_behavior.csv')

In [6]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [7]:
df.info

<bound method DataFrame.info of       Customer ID  Age  Gender Item Purchased     Category  \
0               1   55    Male         Blouse     Clothing   
1               2   19    Male        Sweater     Clothing   
2               3   50    Male          Jeans     Clothing   
3               4   21    Male        Sandals     Footwear   
4               5   45    Male         Blouse     Clothing   
...           ...  ...     ...            ...          ...   
3895         3896   40  Female         Hoodie     Clothing   
3896         3897   52  Female       Backpack  Accessories   
3897         3898   46  Female           Belt  Accessories   
3898         3899   44  Female          Shoes     Footwear   
3899         3900   52  Female        Handbag  Accessories   

      Purchase Amount (USD)       Location Size      Color  Season  \
0                        53       Kentucky    L       Gray  Winter   
1                        64          Maine    L     Maroon  Winter   
2            

In [11]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [16]:
df[ 'Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [17]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [20]:
 df.columns = df.columns.str.lower()
 df.columns = df.columns.str.replace(' ','_')
df=df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [21]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [23]:
# create a column age_group

labels = ['Young Adult' , 'Adult' , 'Middle_aged' , 'Senior']
df['age_group'] = pd.qcut(df['age'] , q=4 , labels = labels)




In [25]:
df[['age' , 'age_group']].head()

,age,age_group
0,55,Middle_aged
1,19,Young Adult
2,50,Middle_aged
3,21,Young Adult
4,45,Middle_aged


In [26]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [27]:
df[['purchase_frequency_days' , 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [30]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [31]:
df=df.drop('promo_code_used',axis=1)

In [32]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [33]:
!pip install pyodbc sqlalchemy 

In [48]:
from sqlalchemy import create_engine
import urllib.parse

# 1. Setup variables
username = r"DESKTOP-7862HL8\DELL SDD"
password = "9106"
host = "localhost" 
database = "customer_behavior"

# 2. URL-encode
safe_user = urllib.parse.quote_plus(username)
safe_password = urllib.parse.quote_plus(password)
driver = urllib.parse.quote_plus("ODBC Driver 17 for SQL Server")

# 3. Create the connection string 

connection_string = f"mssql+pyodbc://@localhost/customer_behavior?driver={driver}&trusted_connection=yes"

# 4. Create engine and upload
engine = create_engine(connection_string)

try:
    table_name = "customer"
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Data successfully loaded into table 'customer' in database 'customer_behavior'.
